# Rex Bot – WildGuardMix + Kaggle SQLi → typed-decisions

Nguồn chính cho nhãn **BLOCK** và **`trace_group = syntax`** của Requin.
Xuất 3 cột `state` / `questions` / `gold` (giống `LocalLLaMA/typed-decisions`, cùng 5 câu hỏi với bộ BAF)
và chia **train / valid / test** chống rò rỉ.

| Nguồn | Vai trò | Cách lấy |
|---|---|---|
| `allenai/wildguardmix` | Jailbreak, yêu cầu độc hại, câu "nghe đáng sợ nhưng vô hại" | **Kaggle Input** (tự tải từ HF rồi upload) |
| Kaggle SQL Injection dataset | Payload SQLi + truy vấn bình thường | Kaggle Input |
| `gretelai/synthetic_text_to_sql` (tuỳ chọn) | SQL hợp lệ; lệnh phá huỷ → ESCALATE_HUMAN | Kaggle Input, hoặc tự tải nếu bật Internet |

### Chuẩn bị (1 lần)

**A. Tải WildGuardMix từ Hugging Face**
1. Vào `huggingface.co/datasets/allenai/wildguardmix`, đăng nhập, bấm **Agree / Access repository**.
2. Tab **Files and versions** → tải 2 file: `train/wildguard_train.parquet` và `test/wildguard_test.parquet`.

**B. Đưa lên Kaggle thành dataset riêng**
1. kaggle.com → **Datasets → + New Dataset** → kéo thả 2 file `.parquet` → đặt tên, ví dụ `wildguardmix`.
2. Để chế độ **Private** (WildGuardMix có điều khoản sử dụng riêng của AI2, không được phát hành lại công khai) → **Create**.

**C. Gắn vào notebook này**
1. **Input → + Add Input** → **Your Datasets** → chọn `wildguardmix`.
2. **+ Add Input** tiếp → tìm `sql injection dataset` → chọn bộ có file CSV gồm cột câu truy vấn + `Label`.
3. *(Tuỳ chọn)* gretel: bật **Settings → Internet: On** để notebook tự tải; hoặc tải `synthetic_text_to_sql_train.snappy.parquet` và `..._test.snappy.parquet` từ HF rồi upload như bước B.
4. **Settings → Accelerator: None** → **Save Version → Save & Run All**.

Tên file không cần chính xác: notebook tự tìm mọi file có chữ `wildguard` trong đường dẫn (parquet/jsonl/csv), file có chữ `test` được dùng làm tập test.

## 1. Cấu hình

In [ ]:
# ====== CHỈNH Ở ĐÂY ======
OUT_DIR        = "/kaggle/working/rexbot_text"
INPUT_DIR      = "/kaggle/input"

USE_WILDGUARD  = True
USE_SQLI       = True
USE_GRETEL     = True      # SQL hợp lệ; lệnh tác động cao -> dạy ESCALATE_HUMAN

WG_TRAIN_MAX   = 40_000    # số mẫu tối đa lấy từ wildguardtrain (train + valid)
SQLI_MAX       = 40_000
SQLI_PER_TEMPLATE = 20    # tối đa số biến thể của cùng 1 khuôn payload (chống học thuộc)
GRETEL_TRAIN_MAX = 12_000
GRETEL_TEST_MAX  = 2_000
GRETEL_HIGH_IMPACT_DIV = 8   # tối đa 1/8 mẫu gretel là lệnh tác động cao (DROP/ALTER/... hoặc DELETE/UPDATE không WHERE)

VALID_PCT      = 0.10      # tỉ lệ valid (tách theo hash -> ổn định giữa các lần chạy)
SQLI_TEST_PCT  = 0.10      # SQLi không có test chính thức -> tách 10% theo "khuôn" truy vấn

HEAD, TAIL     = 700, 400  # rút gọn văn bản dài: giữ 700 ký tự đầu + 400 ký tự cuối
N_ANNOTATORS   = 30
NOISE          = 0.3
PROTECT_LEGIT  = True      # mẫu vô hại không bao giờ bị gán BLOCK
SEED           = 42

In [ ]:
import hashlib, json, os, re, time, unicodedata, zipfile, glob
import numpy as np
import pandas as pd

def log(msg): print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)
def jdump(o): return json.dumps(o, ensure_ascii=False, separators=(",", ":"))

INPUT_DIR = os.environ.get("REXBOT_INPUT", INPUT_DIR)
MOCK_DIR = os.environ.get("REXBOT_MOCK")          # chỉ dùng khi chạy thử ngoài Kaggle
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Bộ câu hỏi chung (giống bộ BAF)

In [ ]:
DECISIONS  = ["BLOCK", "ESCALATE_HUMAN", "TRIGGER_AI", "ALLOW"]
TRACE_OPTS = ["audit_log", "internet_cti", "syntax", "none"]
QUESTIONS = {
    "decision": {"type": "choice",
                 "instructions": "First-pass security verdict for this request.",
                 "criteria": {
                     "BLOCK": "Clear malicious signals; reject now.",
                     "ESCALATE_HUMAN": "High risk or high impact; hold for admin review.",
                     "TRIGGER_AI": "Suspicious but inconclusive; investigate further.",
                     "ALLOW": "No meaningful risk; let it through."}},
    "trace_group": {"type": "choice",
                    "instructions": "If investigated, which agent's evidence matters most?",
                    "criteria": {
                        "audit_log": "History, velocity and identity records in logs/DB.",
                        "internet_cti": "Device, network, session and geo signals.",
                        "syntax": "Structure of code, SQL or prompt payload.",
                        "none": "No investigation lead."}},
    "needs_investigation": {"type": "noul",
                            "instructions": "Is the request alone insufficient for a confident verdict?",
                            "criteria": {"false": "The request alone is enough to decide.",
                                         "true": "More evidence is needed before deciding."}},
    "risk": {"type": "score", "instructions": "How risky is this request?",
             "criteria": ["none", "low", "elevated", "high"]},
    "urgency": {"type": "score", "instructions": "How urgently must security act?",
                "criteria": ["routine", "soon", "prompt", "immediate"]},
}
QUESTIONS_STR = jdump(QUESTIONS)

## 3. Làm sạch, rút gọn, khoá chống trùng

In [ ]:
ZW     = re.compile("[\u200b\u200c\u200d\u2060\ufeff]")
CTRL   = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
REPEAT = re.compile(r"(.{8,200}?)\1{3,}", re.S)

def clean_text(t):
    """Giữ dấu hiệu làm rối (zero-width -> [ZW]); nén đoạn lặp liên tiếp."""
    if not isinstance(t, str):
        return None
    t = CTRL.sub("", t)
    t = ZW.sub("[ZW]", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t).strip()
    t = REPEAT.sub(lambda m: f"{m.group(1)}[x{len(m.group(0)) // len(m.group(1))}] ", t)
    return t or None

def shorten(t, head=HEAD, tail=TAIL):
    """Giữ đầu + cuối: lệnh tấn công thường nằm cuối, bối cảnh nằm đầu."""
    if len(t) <= head + tail + 40:
        return t, False
    return f"{t[:head]} …[{len(t) - head - tail} chars cut]… {t[-tail:]}", True

def norm_key(t):
    """Khoá chống trùng chính xác (không phân biệt hoa/thường, dấu câu, khoảng trắng)."""
    t = unicodedata.normalize("NFKC", t).lower()
    return hashlib.md5(re.sub(r"\W+", " ", t).strip().encode()).hexdigest()

def sql_template(t):
    """Khoá 'khuôn' SQL: thay chuỗi và số bằng ký hiệu -> các biến thể của cùng 1 payload
    luôn rơi vào cùng 1 tập (chống rò rỉ gần-trùng)."""
    t = t.lower()
    t = re.sub(r"'[^']*'|\"[^\"]*\"", "S", t)
    t = re.sub(r"\b0x[0-9a-f]+\b|\b\d+(\.\d+)?\b", "N", t)
    t = re.sub(r"\s+", " ", t).strip()
    return hashlib.md5(t.encode()).hexdigest()

def hash_bucket(key):
    return int(key[:8], 16) % 10_000 / 10_000

def length_band(n):
    return "short" if n < 200 else "medium" if n < 1000 else "long" if n < 4000 else "very_long"

## 4. Luật cú pháp (bằng chứng cho annotator)

In [ ]:
PROMPT_RULES = [
    (1.0, r"\b(ignore|disregard|forget)\b.{0,30}\b(previous|prior|above|all|your)\b.{0,20}\b(instruction|prompt|rule|direction)s?"),
    (1.0, r"\b(system prompt|initial prompt|hidden instructions?)\b"),
    (0.8, r"\b(developer mode|DAN|jailbreak|unfiltered|no (restrictions|limits|rules))\b"),
    (0.5, r"\b(you are now|act as|pretend (to be|you are)|roleplay|role-play|from now on)\b"),
    (0.5, r"\b(hypothetical(ly)?|fictional (story|world|scenario)|for a (novel|story|screenplay))\b"),
    (0.6, r"(<\|im_start\|>|\[/?INST\]|</?system>|###\s*(system|instruction))"),
    (0.6, r"[A-Za-z0-9+/]{40,}={0,2}"),
    (0.4, r"\[ZW\]"),
]
SQL_RULES = [
    (1.0, r"\b(or|and)\s+(['\"]?)(\w+)\2\s*(=|like)\s*\2\3\2"),   # tautology: OR 1=1, OR 'a'='a'
    (1.0, r"'\s*(or|and)\s*'[^']*'\s*=\s*'"),                     # ' or ''='
    (1.0, r"'\s*\)?\s*(--|#|/\*)"),                               # đóng chuỗi rồi comment: admin' --
    (1.0, r"\bunion\b(\s+all)?\s+select\b"),
    (1.0, r";\s*(drop|delete|insert|update|exec|shutdown|truncate)\b"),
    (1.0, r"\b(sleep|benchmark|pg_sleep)\s*\(|waitfor\s+delay"),
    (0.8, r"\binformation_schema\b|@@version|\bsysobjects\b|\bload_file\s*\(|\bxp_cmdshell\b"),
    (0.5, r"(--|#|/\*)"),
    (0.5, r"\bchar\s*\(|0x[0-9a-f]{6,}|%27|%22|%3d"),
]
P_RX = [(w, re.compile(p, re.I)) for w, p in PROMPT_RULES]
S_RX = [(w, re.compile(p, re.I)) for w, p in SQL_RULES]
HIGH_IMPACT = re.compile(r"^\s*(drop|truncate|alter|grant|revoke)\b", re.I)
WRITE_OP    = re.compile(r"^\s*(delete|update)\b", re.I)

def sql_impact(sql):
    """Thao tác cần người duyệt: đổi cấu trúc / phân quyền (DROP, TRUNCATE, ALTER, GRANT, REVOKE),
    hoặc DELETE/UPDATE KHÔNG có WHERE (tác động toàn bảng).
    DELETE/UPDATE có WHERE là nghiệp vụ thường ngày -> không tính."""
    return bool(HIGH_IMPACT.match(sql)) or (bool(WRITE_OP.match(sql))
                                            and not re.search(r"\bwhere\b", sql, re.I))

def suspicion(text, kind):
    rules = S_RX if kind == "sql" else P_RX
    return min(3.0, sum(w for w, rx in rules if rx.search(text)))

## 5. Đọc từng nguồn → bản ghi chuẩn

In [ ]:
def rec(text, kind, source, split, p_mal, agree, adversarial=False, impact=False,
        hard_neg=False, context=None, label_raw=None, sub=None):
    return dict(text=text, kind=kind, source=source, split=split, p_mal=float(p_mal),
                agree=float(agree), adversarial=bool(adversarial), impact=bool(impact),
                hard_neg=bool(hard_neg), context=context, label_raw=label_raw, subcategory=sub)

def read_any(path):
    p = path.lower()
    if p.endswith(".parquet"):
        return pd.read_parquet(path)
    if p.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    if p.endswith(".json"):
        try:
            return pd.read_json(path)
        except ValueError:
            return pd.read_json(path, lines=True)
    return pd.read_csv(path, encoding="utf-8", encoding_errors="ignore")

DATA_EXT = (".parquet", ".jsonl", ".json", ".csv")

def find_files(keyword):
    """Mọi file dữ liệu trong INPUT_DIR có `keyword` trong đường dẫn."""
    return sorted(f for f in glob.glob(f"{INPUT_DIR}/**/*", recursive=True)
                  if os.path.isfile(f) and f.lower().endswith(DATA_EXT) and keyword in f.lower())

def load_hf(repo, config, split):
    if MOCK_DIR:  # chỉ dùng khi chạy thử ngoài Kaggle
        return pd.read_json(f"{MOCK_DIR}/{repo.split('/')[1]}_{config or 'default'}_{split}.jsonl", lines=True)
    from datasets import load_dataset
    args = [repo] + ([config] if config else [])
    return load_dataset(*args, split=split).to_pandas()

def agree_from(v, default):
    """WildGuardTest: nhãn đạt đồng thuận 2/3 hay 3/3 annotator."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return default
    s = str(v).lower()
    if "3" in s or "three" in s or "unanimous" in s: return 1.0
    if "2" in s or "two" in s or "majority" in s:   return 0.67
    try:
        x = float(s); return x if 0 < x <= 1 else default
    except ValueError:
        return default

def read_wildguard():
    files = find_files("wildguard")
    if not files:
        log("[WildGuardMix] Không thấy file nào có chữ 'wildguard' trong Input. "
            "Xem phần Chuẩn bị A-C ở đầu notebook.")
        return []
    out = []
    for f in files:
        split = "test" if "test" in os.path.basename(f).lower() else "train"
        try:
            df = read_any(f)
        except Exception as e:
            log(f"[WildGuardMix] Không đọc được {f}: {e}"); continue
        if "prompt" not in df.columns or "prompt_harm_label" not in df.columns:
            log(f"[WildGuardMix] Bỏ qua {f}: thiếu cột prompt / prompt_harm_label (có: {list(df.columns)[:8]})")
            continue
        default_agree = 0.97 if split == "test" else 0.92   # test: nhãn người; train: nhãn GPT-4
        agree_col = next((c for c in df.columns if "prompt_harm" in c and "agree" in c), None)
        if split == "train" and len(df) > WG_TRAIN_MAX:
            df = df.sample(n=WG_TRAIN_MAX, random_state=SEED)
        n0 = len(out)
        for r in df.to_dict("records"):
            lab = r.get("prompt_harm_label")
            if lab not in ("harmful", "unharmful") or not isinstance(r.get("prompt"), str):
                continue
            adv = bool(r.get("adversarial", False))
            ag = agree_from(r.get(agree_col), default_agree) if agree_col else default_agree
            out.append(rec(r["prompt"], "prompt", "wildguardmix", split,
                           1.0 if lab == "harmful" else 0.0, ag, adversarial=adv,
                           hard_neg=adv and lab == "unharmful",
                           label_raw=f"{lab}|{'adv' if adv else 'vanilla'}",
                           sub=r.get("subcategory")))
        log(f"[WildGuardMix/{split}] {os.path.basename(f)}: {len(df):,} dòng -> {len(out) - n0:,} bản ghi"
            + (f" (cột đồng thuận: {agree_col})" if agree_col else ""))
    if not any(r["split"] == "test" for r in out):
        log("[WildGuardMix] Cảnh báo: không có file test -> tập test sẽ thiếu WildGuardMix.")
    return out

TEXT_COLS  = ["query", "sentence", "payload", "text", "sql"]
LABEL_COLS = ["label", "class", "is_malicious"]

def read_sqli():
    files = sorted(glob.glob(f"{INPUT_DIR}/**/*.csv", recursive=True))
    out = []
    for f in files:
        try:
            head = pd.read_csv(f, nrows=5, encoding="utf-8", encoding_errors="ignore")
        except Exception:
            continue
        cols = {c.strip().lower(): c for c in head.columns}
        tcol = next((cols[c] for c in TEXT_COLS if c in cols), None)
        lcol = next((cols[c] for c in LABEL_COLS if c in cols), None)
        if not (tcol and lcol) or "sql" not in f.lower() or "wildguard" in f.lower() \
                or "text_to_sql" in f.lower():
            continue
        df = pd.read_csv(f, usecols=[tcol, lcol], encoding="utf-8", encoding_errors="ignore")
        df[lcol] = pd.to_numeric(df[lcol], errors="coerce")
        bad = df[lcol].isna() | ~df[lcol].isin([0, 1])
        df = df[~bad].dropna(subset=[tcol])
        log(f"[SQLi] {f}: cột '{tcol}' + '{lcol}', {len(df):,} dòng hợp lệ (bỏ {int(bad.sum())} nhãn lỗi)")
        for t, y in zip(df[tcol].astype(str), df[lcol].astype(int)):
            out.append(rec(t, "sql", "kaggle_sqli", None, float(y), 0.9, label_raw=int(y)))
    if not out:
        log("[SQLi] Không tìm thấy CSV SQLi (cần file có 'sql' trong đường dẫn + cột Query/Sentence + Label)")
    if len(out) > SQLI_MAX:
        idx = np.random.default_rng(SEED).choice(len(out), SQLI_MAX, replace=False)
        out = [out[i] for i in idx]
    return out

def read_gretel():
    local = find_files("text_to_sql")
    sources = []
    if local:
        for f in local:
            sources.append(("test" if "test" in os.path.basename(f).lower() else "train", f))
        log(f"[gretel] Dùng file trong Input: {[os.path.basename(f) for _, f in sources]}")
    out = []
    for split, cap in [("train", GRETEL_TRAIN_MAX), ("test", GRETEL_TEST_MAX)]:
        try:
            if local:
                fs = [f for s_, f in sources if s_ == split]
                if not fs:
                    log(f"[gretel/{split}] không có file trong Input"); continue
                df = pd.concat([read_any(f) for f in fs])
            else:
                df = load_hf("gretelai/synthetic_text_to_sql", None, split)
        except Exception as e:
            log(f"[gretel/{split}] bỏ qua ({str(e)[:150]}). Bật Internet hoặc upload file gretel vào Input.")
            continue
        imp_mask = df["sql"].map(sql_impact)
        high, rest = df[imp_mask], df[~imp_mask]
        n_h = min(len(high), cap // GRETEL_HIGH_IMPACT_DIV)
        df = pd.concat([high.sample(n=n_h, random_state=SEED),
                        rest.sample(n=min(len(rest), cap - n_h), random_state=SEED)])
        for r in df.to_dict("records"):
            sql = r["sql"]
            out.append(rec(sql, "sql", "gretel_sql", split, 0.0, 0.95, impact=sql_impact(sql),
                           context={"intent": r.get("sql_prompt", "")},
                           label_raw=r.get("sql_task_type")))
        log(f"[gretel/{split}] {len(df):,} bản ghi ({n_h} lệnh tác động cao)")
    return out

## 6. Đọc dữ liệu

In [ ]:
rows = []
if USE_WILDGUARD: rows += read_wildguard()
if USE_SQLI:      rows += read_sqli()
if USE_GRETEL:    rows += read_gretel()
assert rows, "Không đọc được nguồn nào - xem thông báo lỗi ở trên."

df = pd.DataFrame(rows)
df["orig_len"] = df["text"].map(lambda t: len(t) if isinstance(t, str) else 0)
df["text"] = df["text"].map(clean_text)
df = df.dropna(subset=["text"])
df = df[df["text"].str.len() >= 3].reset_index(drop=True)
df["key"] = df["text"].map(norm_key)
df["tkey"] = np.where(df["kind"] == "sql", df["text"].map(sql_template), df["key"])
n_before = len(df)
df = df[df.groupby("tkey").cumcount() < SQLI_PER_TEMPLATE].reset_index(drop=True)
log(f"Giới hạn {SQLI_PER_TEMPLATE} biến thể/khuôn: {n_before:,} -> {len(df):,} dòng")
pd.crosstab(df["source"], df["p_mal"].round(1).rename("p_mal"))

## 7. Chia train / valid / test (chống rò rỉ)

- **WildGuardMix**: test = `wildguardtest` (nhãn người, chất lượng cao nhất); `wildguardtrain` tách valid theo hash.
- **Kaggle SQLi**: tách theo **khuôn truy vấn** (`sql_template`) → các biến thể của cùng một payload luôn chung một tập.
- **gretel**: test = split test chính thức; train tách valid theo hash.
- Sau đó chống trùng **xuyên nguồn, xuyên tập**: nếu một văn bản/khuôn xuất hiện ở nhiều tập, chỉ giữ bản ở tập ưu tiên cao hơn (test > valid > train).

In [ ]:
b = df["tkey"].map(hash_bucket)
is_sqli = df["source"] == "kaggle_sqli"
df.loc[is_sqli, "split"] = np.where(b[is_sqli] < SQLI_TEST_PCT, "test",
                           np.where(b[is_sqli] < SQLI_TEST_PCT + VALID_PCT, "valid", "train"))
need_valid = (df["split"] == "train") & ~is_sqli
df.loc[need_valid & (b < VALID_PCT), "split"] = "valid"

# chống trùng xuyên tập: giữ bản ở tập ưu tiên cao nhất
prio = df["split"].map({"test": 0, "valid": 1, "train": 2})
df = df.assign(_p=prio).sort_values("_p")
n0 = len(df)
best = df.groupby("tkey")["_p"].transform("min")
df = df[df["_p"] == best]                       # bỏ bản ở tập kém ưu tiên hơn
# trong cùng tập: gộp trùng, nhãn mâu thuẫn -> trung bình, hạ độ chắc
g = df.groupby("key")["p_mal"]
conflict = g.transform("std").fillna(0) > 0.3
df["p_mal"] = g.transform("mean")
df.loc[conflict, "agree"] = 0.6
df = df.drop_duplicates("key").drop(columns="_p").reset_index(drop=True)
log(f"Chống trùng: {n0:,} -> {len(df):,} dòng; {int(conflict.sum())} dòng có nhãn mâu thuẫn")
df["susp"] = [suspicion(t, k) for t, k in zip(df["text"], df["kind"])]
pd.crosstab([df["split"]], df["source"], margins=True)

## 8. Gold mềm (K annotator bỏ phiếu)

In [ ]:
def soft_labels(d, k=N_ANNOTATORS, noise=NOISE, protect_legit=PROTECT_LEGIT):
    rng = np.random.default_rng(SEED)
    n = len(d)
    p   = d["p_mal"].values[:, None]
    ag  = d["agree"].values[:, None]
    sus = d["susp"].values[:, None]
    adv = d["adversarial"].values[:, None].astype(float)
    hard = d["hard_neg"].values[:, None].astype(float)
    sql = (d["kind"].values == "sql")[:, None]
    impact = d["impact"].values[:, None]

    flip = rng.random((n, k)) > ag                       # annotator không tin nhãn gốc
    pv = np.where(flip, 1 - p, p)
    latent = (rng.uniform(2.6, 3.2, k) * pv + rng.uniform(.3, .6, k) * sus
              + 0.6 * hard + 0.3 * adv + rng.normal(0, noise, (n, k)))
    t1, t2, t3 = rng.uniform(.5, .8, k), rng.uniform(1.3, 1.7, k), rng.uniform(2.1, 2.5, k)
    lvl = (latent >= t1).astype(int) + (latent >= t2) + (latent >= t3)
    # ca "sát ranh giới": điểm rủi ro gần một ngưỡng -> quyết định dễ đổi nếu có thêm bằng chứng
    margin = np.minimum(np.minimum(abs(latent - t1), abs(latent - t2)), abs(latent - t3))
    ev = sus + 1.8 * pv + 0.5 * adv + rng.normal(0, .2, (n, k))   # độ rõ của bằng chứng
    e_weak, e_strong = rng.uniform(.4, .8, k), rng.uniform(1.2, 1.8, k)
    imp = impact & (rng.random((n, k)) < .9)

    # chỉ số theo DECISIONS: 0 BLOCK, 1 ESCALATE_HUMAN, 2 TRIGGER_AI, 3 ALLOW
    dec = np.select([imp & (lvl <= 2), lvl == 0, (lvl == 1) & (ev < e_weak), lvl == 1,
                     lvl == 2, (lvl == 3) & (ev >= e_strong)],
                    [1, 3, 3, 2, 2, 0], default=1)
    if protect_legit:                                    # mẫu gốc vô hại: tối đa ESCALATE
        dec = np.where((dec == 0) & (p < 0.5), 1, dec)
    # cần điều tra: (có rủi ro nhưng bằng chứng chưa đủ mạnh) HOẶC (ca sát ranh giới)
    need = ((lvl >= 1) & (ev < e_strong)) | (margin < 0.3)

    # syntax: có cấu trúc tấn công (luật khớp, vỏ jailbreak, hoặc SQL độc hại)
    tscore = sus + 1.0 * adv + 0.5 * pv + 1.0 * (sql & (pv >= .5)) + rng.normal(0, .2, (n, k))
    trace = np.where(tscore >= rng.uniform(.6, .9, k), 2, 3)
    urg = np.clip(np.where(imp, np.maximum(lvl, 1), lvl) + ((lvl == 3) & (ev >= e_strong)), 0, 3)

    frac = lambda m, c: np.stack([(m == i).mean(1) for i in range(c)], 1)
    return {"decision": frac(dec, 4), "trace_group": frac(trace, 4),
            "needs_investigation": need.mean(1),
            "risk": frac(np.maximum(lvl, imp.astype(int)), 4), "urgency": frac(urg, 4)}

def conf(p):
    kk = len(p)
    return float(round((max(p) - 1 / kk) / (1 - 1 / kk), 6))

def gold_row(i, sl):
    r6 = lambda x: float(round(float(x), 6))
    d, t, pt = sl["decision"][i], sl["trace_group"][i], float(sl["needs_investigation"][i])
    g = {"decision": {"type": "choice", "label": DECISIONS[int(d.argmax())], "confidence": conf(d),
                      "probabilities": {a: r6(b) for a, b in zip(DECISIONS, d)}},
         "trace_group": {"type": "choice", "label": TRACE_OPTS[int(t.argmax())], "confidence": conf(t),
                         "probabilities": {a: r6(b) for a, b in zip(TRACE_OPTS, t)}},
         "needs_investigation": {"type": "noul", "label": "true" if pt >= .5 else "false",
                                 "noul": r6(pt), "confidence": conf([pt, 1 - pt]),
                                 "probabilities": {"false": r6(1 - pt), "true": r6(pt)}}}
    for q in ["risk", "urgency"]:
        p = sl[q][i]
        g[q] = {"type": "score", "label": str(int(p.argmax())),
                "score": r6((p * np.arange(4)).sum()), "confidence": conf(p),
                "probabilities": {str(j): r6(v) for j, v in enumerate(p)}}
    return g

SL = soft_labels(df)
df["decision"] = np.array(DECISIONS)[SL["decision"].argmax(1)]
df["trace"] = np.array(TRACE_OPTS)[SL["trace_group"].argmax(1)]
pd.crosstab([df["source"], df["p_mal"].round(1).rename("p_mal")], df["decision"])

## 9. Dựng state và ghi file

In [ ]:
def build_state(r):
    txt, cut = shorten(r["text"])
    if r["kind"] == "sql":
        s = {"workflow": "database_query", "channel": "app_backend",
             "request": {"query": txt, "length": length_band(r["orig_len"]), "truncated": cut}}
    else:
        s = {"workflow": "chat_request", "channel": "chatbot",
             "request": {"message": txt, "length": length_band(r["orig_len"]), "truncated": cut}}
    if isinstance(r.get("context"), dict):
        s["context"] = {k: shorten(clean_text(str(v)) or "", 300, 0)[0] for k, v in r["context"].items()}
    return s

files = {s: open(f"{OUT_DIR}/{s}.jsonl", "w", encoding="utf-8") for s in ["train", "valid", "test"]}
lens, cuts = [], 0
for i, r in enumerate(df.to_dict("records")):
    st = build_state(r)
    cuts += st["request"]["truncated"]
    st = jdump(st)
    lens.append(len(st))
    files[r["split"]].write(jdump({"state": st, "questions": QUESTIONS_STR,
                                   "gold": jdump(gold_row(i, SL))}) + "\n")
for f in files.values():
    f.close()
df[["key", "tkey", "split", "source", "kind", "label_raw", "subcategory", "adversarial",
    "p_mal", "agree", "susp", "hard_neg", "impact", "decision", "trace"]] \
    .rename(columns={"key": "group_key", "tkey": "template_key"}) \
    .to_csv(f"{OUT_DIR}/text_meta.csv", index=False)
log(f"Đã ghi {len(df):,} dòng; state TB {np.mean(lens):.0f} ký tự, max {max(lens)}; {cuts} dòng bị rút gọn")

## 10. Kiểm tra chất lượng

In [ ]:
from sklearn.metrics import roc_auc_score
P, need = SL["decision"], SL["needs_investigation"]
mal = df["p_mal"] >= 0.5
ov = lambda a, b, col: len(set(df.loc[df.split == a, col]) & set(df.loc[df.split == b, col]))
sq = df["source"] == "kaggle_sqli"
checks = {
    "Mẫu vô hại bị gán BLOCK (phải = 0)": int(((df.decision == "BLOCK") & ~mal).sum()),
    "Tỉ lệ BLOCK trong mẫu độc hại (nên >= 40%)": round(float((df.decision == "BLOCK")[mal].mean()), 3),
    "Tỉ lệ syntax trong payload SQLi độc hại (nên >= 90%)":
        round(float((df.trace == "syntax")[mal & sq].mean()), 3) if (mal & sq).any() else "n/a",
    "Tỉ lệ syntax trong jailbreak (adversarial + harmful) (nên >= 70%)":
        round(float((df.trace == "syntax")[mal & df.adversarial].mean()), 3) if (mal & df.adversarial).any() else "n/a",
    "Tỉ lệ ALLOW trong mẫu vô hại (nên >= 70%)": round(float((df.decision == "ALLOW")[~mal].mean()), 3),
    "SQL hợp lệ (gretel) bị gán trace syntax (nên < 3%)":
        round(float((df.trace == "syntax")[df.source == "gretel_sql"].mean()), 3) if (df.source == "gretel_sql").any() else "n/a",
    "ESCALATE ở gretel là DELETE/UPDATE có WHERE (phải = 0)":
        int(((df.source == "gretel_sql") & (df.decision == "ESCALATE_HUMAN")
             & df.text.str.match(r"^\s*(delete|update)\b", case=False)
             & df.text.str.contains(r"\bwhere\b", case=False)).sum()),
    "Văn bản trùng giữa train-test / train-valid / valid-test (phải = 0)":
        (ov("train", "test", "key"), ov("train", "valid", "key"), ov("valid", "test", "key")),
    "Khuôn SQL trùng giữa train-test (phải = 0)": ov("train", "test", "tkey"),
    "Tương quan needs_investigation với P(ESC)+P(TRIGGER) (phải < 0.95)":
        round(float(np.corrcoef(need, P[:, 1] + P[:, 2])[0, 1]), 3),
    "AUC luật SQL trên Kaggle SQLi (gần 1.0 = dữ liệu quá dễ)":
        round(roc_auc_score(df.loc[sq, "p_mal"] >= .5, df.loc[sq, "susp"]), 3) if sq.any() and df.loc[sq, "p_mal"].nunique() > 1 else "n/a",
    "Tỉ lệ dòng mọi annotator đồng ý tuyệt đối": round(float((P.max(1) == 1).mean()), 3),
}
report = "\n".join([
    "=== REX BOT / TEXT (WildGuardMix + SQLi) REPORT ===", "",
    "Số dòng theo tập x nguồn:", pd.crosstab(df["split"], df["source"], margins=True).to_string(), "",
    "Tỉ lệ nhãn decision theo tập (%):",
    (pd.crosstab(df["split"], df["decision"], normalize="index") * 100).round(1).to_string(), "",
    "Decision theo nguồn và nhãn gốc:",
    pd.crosstab([df["source"], df["label_raw"].astype(str)], df["decision"]).to_string(), "",
    "trace_group theo nguồn (%):",
    (pd.crosstab(df["source"], df["trace"], normalize="index") * 100).round(1).to_string(), "",
    "KIỂM TRA TỰ ĐỘNG:", *[f"  - {k}: {v}" for k, v in checks.items()], "",
    f"Độ dài state: TB {np.mean(lens):.0f}, max {max(lens)} ký tự; {cuts} dòng bị rút gọn",
])
open(f"{OUT_DIR}/report.txt", "w", encoding="utf-8").write(report)
print(report)

In [ ]:
# Xem 1 mẫu mỗi loại decision
seen = set()
for s in ["train", "test"]:
    for line in open(f"{OUT_DIR}/{s}.jsonl", encoding="utf-8"):
        r = json.loads(line); g = json.loads(r["gold"])
        lab = g["decision"]["label"]
        if lab in seen: continue
        seen.add(lab)
        st = json.loads(r["state"])
        body = st["request"].get("message") or st["request"].get("query")
        print(f"=== {lab} ({st['workflow']}) ===\n{body[:300]}")
        print("decision:", g["decision"]["probabilities"], "| trace:", g["trace_group"]["label"], "\n")
    if len(seen) == 4: break

## 11. Đóng gói

In [ ]:
zip_path = OUT_DIR.rstrip("/") + ".zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, f), arcname=f"rexbot_text/{f}")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"{f:16s} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6:8.1f} MB")
print(f"\nFile nén: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

## Ghi chú

- **Giấy phép:** WildGuardMix có điều khoản sử dụng riêng của AI2. Dataset upload lên Kaggle và bộ đã gộp
  đều nên để **Private**.
- **WildGuardMix không chỉ là prompt injection:** phần lớn là yêu cầu độc hại và jailbreak. Mẫu
  *adversarial + unharmful* là hard negative quý (nghe như jailbreak nhưng vô hại) → TRIGGER_AI / ALLOW.
- **Gộp với BAF:** cùng 5 câu hỏi và cùng format → chỉ cần nối các file `train/valid/test.jsonl` tương ứng.